# PV + CSP + BES + TES

A standalone microgrid demonstration. It uses 5% PNM demand, 3,000× PV availability, unscaled CSP, a 750 MWh / 150 MW battery, and 2,500 MWh-thermal / 150 MW TES.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

root = Path.cwd()
while not (root / 'enliten').is_dir():
    if root.parent == root: raise RuntimeError('Run from inside the ENLITEN repository.')
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from enliten import ChargingPath, Generation, LCOECalculator, Site, Storage, System
data_dir = root / 'examples' / 'data'

def profile(filename):
    frame = pd.read_csv(data_dir / filename)
    return pd.Series(frame['PNM'].to_numpy(float), index=pd.to_datetime(frame.iloc[:, 0], utc=True))

demand_full = profile('PNM_demand.csv')
pv_full = profile('PNM_pv_ac_1MW_av.csv')
csp_full = profile('PNM_csp_th_av.csv')
assert demand_full.index.equals(pv_full.index) and demand_full.index.equals(csp_full.index)

def build_system(start, hours=24 * 14):
    window = demand_full.index.get_loc(pd.Timestamp(start, tz='UTC'))
    load = (demand_full.iloc[window:window + hours] * 0.05).rename('load_MW')
    pv_multiplier, bes_capacity_MWh, bes_power_MW = 3_000.0, 750.0, 150.0
    tes_capacity_MWh_th, tes_power_MW_e = 2_500.0, 150.0
    pv_capex = pv_full.max() * pv_multiplier * 1_000 * 1_430
    pv_opex = pv_full.max() * pv_multiplier * 1_000 * 24
    bes_capex = bes_capacity_MWh * 1_000 * 300
    csp_tes_capex = tes_power_MW_e * 1_000 * 7_912
    site = Site('microgrid')
    csp = Generation('csp', site, csp_full.iloc[window:window + hours], 'thermal', False, False, capex=csp_tes_capex, opex=tes_power_MW_e * 1_000 * 74.6)
    pv = Generation('pv', site, pv_full.iloc[window:window + hours] * pv_multiplier, 'electric', capex=pv_capex, opex=pv_opex)
    tes = Storage('tes', site, tes_capacity_MWh_th, tes_power_MW_e, 'thermal', 'electric', 0.50, maximum_stored_energy_rate_MW=300.0, variable_opex_USD_per_MWh=3.8)
    bes = Storage('bes', site, bes_capacity_MWh, bes_power_MW, 'electric', 'electric', 0.90, maximum_stored_energy_rate_MW=bes_power_MW, capex=bes_capex, opex=0.025 * bes_capex)
    paths = [ChargingPath('csp', 'tes', 'thermal', 'thermal', 0.90, 300.0 / 0.90), ChargingPath('pv', 'bes', 'electric', 'electric', 0.90, bes_power_MW / 0.90)]
    return System(load, [tes, csp, bes, pv], paths)

system = build_system('2023-01-01')
system.timeseries.head()

In [ ]:
system.operation_metrics()

In [ ]:
LCOECalculator.from_system(system).calculate_lcoe_metrics()

In [ ]:
summer_system = build_system('2023-07-01')
def fully_served_hours(candidate):
    return 100 * candidate.timeseries.iloc[1:]['grid_to_load_MWh_electric'].lt(1e-9).mean()
pd.Series({'January fully served hours (%)': fully_served_hours(system), 'July fully served hours (%)': fully_served_hours(summer_system)}).round(1)

In [ ]:
fig, ax = system.timeseries_plot_source(start_date=0, days=2)
fig

In [ ]:
fig, ax = system.timeseries_plot_group(start_date=0, days=2)
fig

In [ ]:
fig, ax = system.plot_storage_capacity(start_date=0, days=2)
fig

In [ ]:
system.resilience_cases(critical_load_MW=20.0, target_hours=24, n_starts=20, seed=7)
system.resilience_summary